# World Cup 2026 — Group Stage Simulation

Monte Carlo simulation of the WC 2026 group stage. Swap the `predictor` variable to try different models.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from worldcup.tournament import load_teams, get_groups
from worldcup.simulator import GroupStageSimulator
from worldcup.algorithms import RankingPredictor, UniformPredictor

sns.set_theme(style='whitegrid', palette='muted')

In [ ]:
# --- Configuration ---
predictor = RankingPredictor()   # swap to UniformPredictor() or your own
N_SIMULATIONS = 20_000
SEED = 42

teams = load_teams()
groups = get_groups(teams)
flag_map = {t.name: t.flag for t in teams}
team_to_group = {t.name: t.group for t in teams}

print(f'Running {N_SIMULATIONS:,} simulations with model: {predictor.name}')
sim = GroupStageSimulator(predictor, n=N_SIMULATIONS, seed=SEED)
results = sim.run(groups)
df = results.to_dataframe()

df['flag']  = df['team'].map(flag_map)
df['group'] = df['team'].map(team_to_group)
# label: emoji + name for HTML tables; plain team name for matplotlib axes
df['label'] = df['flag'] + ' ' + df['team']

print('Done.')
df.head(10)

## Qualification probabilities

In [ ]:
fig, ax = plt.subplots(figsize=(12, 14))

palette = sns.color_palette('tab20', n_colors=12)
group_colours = {g: palette[i] for i, g in enumerate(sorted(team_to_group.values()))}
colours = [group_colours[g] for g in df['group']]

# Plain team names on chart axes — emoji don't render in matplotlib PNG output
ax.barh(df['team'], df['p_qualify'], color=colours)
ax.set_xlabel('Probability of qualifying from group stage')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title(f'WC 2026 group stage qualification — model: {predictor.name} ({N_SIMULATIONS:,} sims)', fontsize=13)
ax.invert_yaxis()

from matplotlib.patches import Patch
handles = [Patch(color=group_colours[g], label=f'Group {g}') for g in sorted(group_colours)]
ax.legend(handles=handles, loc='lower right', ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

## Breakdown by outcome (1st / 2nd / best 3rd)

In [ ]:
top_n = 20
top = df.head(top_n).copy()

fig, ax = plt.subplots(figsize=(12, 8))

ax.barh(top['team'], top['p_group_winner'], label='1st in group', color='#2196F3')
ax.barh(top['team'], top['p_runner_up'], left=top['p_group_winner'], label='2nd in group', color='#4CAF50')
ax.barh(top['team'], top['p_best_third'], left=top['p_group_winner'] + top['p_runner_up'], label='Best 3rd', color='#FF9800')

ax.set_xlabel('Probability')
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))
ax.set_title(f'Top {top_n} teams — qualification route breakdown', fontsize=13)
ax.legend()
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Group-level heatmap

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, (group_name, group_teams) in enumerate(sorted(groups.items())):
    group_df = df[df['group'] == group_name].copy()
    ax = axes[i]

    cols = ['p_group_winner', 'p_runner_up', 'p_best_third', 'p_eliminated']
    col_labels = ['1st', '2nd', 'Best 3rd', 'Out']
    heatmap_data = group_df.set_index('team')[cols]
    heatmap_data.columns = col_labels

    sns.heatmap(
        heatmap_data,
        ax=ax,
        annot=True,
        fmt='.0%',
        vmin=0,
        vmax=1,
        cmap='YlOrRd',
        cbar=False,
        linewidths=0.5,
    )
    ax.set_title(f'Group {group_name}', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.suptitle(f'WC 2026 group stage probabilities — {predictor.name}', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Results table

In [ ]:
# Emoji render fine in HTML/pandas display
display_df = df[['flag', 'group', 'team', 'p_qualify', 'p_group_winner', 'p_runner_up', 'p_best_third', 'p_eliminated']].copy()
display_df['Team'] = display_df['flag'] + ' ' + display_df['team']
display_df = display_df[['group', 'Team', 'p_qualify', 'p_group_winner', 'p_runner_up', 'p_best_third', 'p_eliminated']]
display_df.columns = ['Group', 'Team', 'Qualify', '1st', '2nd', 'Best 3rd', 'Eliminated']

display_df.style \
    .hide(axis='index') \
    .format({c: '{:.1%}' for c in ['Qualify', '1st', '2nd', 'Best 3rd', 'Eliminated']}) \
    .background_gradient(subset='Qualify',    cmap='RdYlGn',   vmin=0, vmax=1) \
    .background_gradient(subset='Eliminated', cmap='RdYlGn_r', vmin=0, vmax=1)